In [3]:
# Importing relevant packages, modules, and functions

import KTBoost.KTBoost as KTBoost
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize, BFGS
from numpy.polynomial.hermite import hermgauss
from scipy.special import logsumexp
from scipy.stats import chi2, norm
from numpy.linalg import inv, pinv
import time
import copy
from joblib import Parallel, delayed
import warnings
from sklearn.exceptions import ConvergenceWarning
from collections import Counter

# 1. Code for frailty models (time-independent frailties) and simulation study

###  1.1. Implementation of the objective function (negative marginal log-likelihood) using GHQ integration: Case of time-independent frailties
- See section 2.5.2.3
- In the codes 'individual' and 'group' mean the same thing
- We minimise the negatove log-likelihood, hence the minus at in front of the negative log-likelihood

In [2]:
def loglik_glmm_ghq_vectorized(params, X, y, groups, n_individuals, n_quad=20):

    beta = params[:-1]       # Extracting fixed effects
    log_sigma_u = params[-1]        # Taking log of frailty (i.e., std dev of frailty variance) - 
                                    # We use log(sigma_u) as input as it helps with unconstrained optimisation
    
    sigma_u = np.exp(log_sigma_u)       # Taking log back to make positive

    # Gauss-Hermite quadrature
    ghq_points, ghq_weights = hermgauss(n_quad)
    ghq_points = ghq_points[:, None] * np.sqrt(2) * sigma_u
    ghq_weights = ghq_weights / np.sqrt(np.pi)  # Normalisation

    # Linear predictors and response
    eta_fixed = np.asarray(X @ beta)[:, None]         # Shape (n_obs, 1)
    y = np.asarray(y)

    # Broadcast quadrature points
    eta_all = eta_fixed + ghq_points.T                # Shape (n_obs, n_quad)
    p_all = 1 / (1 + np.exp(-np.clip(eta_all, -30, 30)))  # Sigmoid function (clipped to avoid numerical problems)

    # vectorised log-likelihoods
    log_likelihoods = y[:, None] * np.log(p_all + 1e-8) + (1 - y[:, None]) * np.log(1 - p_all + 1e-8)

    # Safety check
    if np.max(groups) >= n_individuals:
        raise ValueError("Group indices exceed declared number of individuals. Check group encoding.")

    # Aggregating per group (i.e. individual)
    loglik_per_observation = np.zeros((n_individuals, n_quad))
    np.add.at(loglik_per_observation, groups, log_likelihoods)

    # Integrating with log-sum-exp
    loglik_per_individual = logsumexp(loglik_per_observation, b=ghq_weights, axis=1)

    return -np.sum(loglik_per_individual)    


###  1.2.  Implementation of the objective function (expected complete-data negative log-likelihood) using the an adaptation of the EM algorithm  (see section 2.5.3.3.1) of my thesis

In [3]:
# def em_logistic_glmm(X, y, groups, n_quad=20, max_iter=1000, tol=1e-5, verbose=True):
def em_logistic_glmm_check(X, y, groups, n_quad=20, max_iter=1000, tol=1e-5, verbose=False):

    # Initialisation of dimensions
    n_obs, p = X.shape
    unique_groups, group_idx = np.unique(groups, return_inverse=True)
    n_groups = len(unique_groups)

    # Setting Initial values of beta through logistic regression (LLink)
    init_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
    init_model.fit(X, y)
    beta = init_model.coef_.flatten()

    # Initialising sigma_u using residual values
    eta = X @ beta
    sigma_u = np.std(eta - eta.mean())

    # Gauss-Hermite quadrature nodes and weights
    gh_x, gh_w = hermgauss(n_quad)
    gh_x = gh_x * np.sqrt(2)
    gh_w = gh_w / np.sqrt(np.pi)

    loglik_trace = []
    start_time = time.time()
    

    for iteration in range(max_iter):
        # ----- E-Step -----
        # Computing linear predictors (vectorised)
        eta_fixed = X @ beta
        eta_fixed_q = np.repeat(eta_fixed[:, None], n_quad, axis=1)  # (n_obs, n_quad)
        u_q = sigma_u * gh_x  # (n_quad,)
        eta_all = eta_fixed_q + u_q  # vectorising to (n_obs, n_quad)
        eta_all = np.clip(eta_all, -30, 30)

        # Computing probabilities with LLink 
        p_all = 1 / (1 + np.exp(-eta_all))  # (n_obs, n_quad)

        # Compute log-likelihood contributions: (n_obs, n_quad)
        ll_obs_q = y[:, None] * np.log(p_all + 1e-8) + (1 - y[:, None]) * np.log(1 - p_all + 1e-8)

        # Suming within each group
        ll_group_q = np.zeros((n_groups, n_quad))
        np.add.at(ll_group_q, group_idx, ll_obs_q)

        # Computing log p(y_i | u_q) + log g(u_q)
        
        log_weights = ll_group_q + (-0.5 * (u_q**2))[None, :]  # (n_groups, n_quad) | This computes the numerator in log-space
        log_weights = log_weights - logsumexp(log_weights, axis=1, keepdims=True)  # normalisation | This line subtracts the 
#         log-sum-exp over quadrature nodes q for each individual i
        weights = np.exp(log_weights)  # posterior weights w_{iq} | In general this is what is being computed :
#          exp(log(x) - logsumexp(x)). This helps with numerical stability

        # ----- M-Step -----
        # Setting up the objective function for the expected complete-data negative log-likelihood
        def mstep_objective(params):
            beta_new = params[:-1]
            sigma_log = params[-1]
            sigma_new = np.exp(sigma_log)

            eta = X @ beta_new
            eta_q = eta[:, None] + sigma_new * gh_x[None, :]
            eta_q = np.clip(eta_q, -30, 30)
            p_q = 1 / (1 + np.exp(-eta_q))

            ll_obs_q = y[:, None] * np.log(p_q + 1e-8) + (1 - y[:, None]) * np.log(1 - p_q + 1e-8)

            ll_group_q = np.zeros((n_groups, n_quad)) 
            np.add.at(ll_group_q, group_idx, ll_obs_q)  

            # Expected log-likelihood
            expected_ll = np.sum(weights * ll_group_q) # First part of integral 
            # Adding E[log g(u)] term
            expected_ll += -0.5 * np.sum(weights * (gh_x[None, :]**2)) # Adding first to second part of integral 

            return -expected_ll  # minimising expected complete-data negative log-likelihood

        # Optimisation of the parameters in the M-step
        res = minimize(
            mstep_objective,
            x0=np.append(beta, np.log(sigma_u)),
            method='L-BFGS-B',
            options={'verbose':1}
        )

        # Updating parameters
        beta_new = res.x[:-1]
        sigma_new = np.exp(res.x[-1])
        loglik_trace.append(-res.fun)

        # Convergence check
        if iteration > 0:
            if np.linalg.norm(beta_new - beta) < tol and abs(sigma_new - sigma_u) < tol:
                if verbose:
                    print(f"✅ Converged at iteration {iteration}")
                break

        beta = beta_new
        sigma_u = sigma_new
        if verbose:
            print(f"Iter {iteration}: sigma_u = {sigma_u:.4f}, loglik = {loglik_trace[-1]:.4f}")
            
        if time.time() - start_time > 120:  # every 2 minutes
            print(f"Iteration {iteration}: beta norm = {np.linalg.norm(beta)}, sigma_u = {sigma_u:.4f}")
            start_time = time.time()

    return beta, sigma_u, loglik_trace, res


### 1.3 Code for simulation study

In [4]:
# List to save final estimates from the simulation study below
B_Res_simul_synth3 = []


# Setting up the list of (true) fixed effects parameters
vec_beta_true=[np.array([0.5, 0.5, -1.2]), np.array([1.5, -0.8, 1.3]), np.array([-1.438,  4.847, -0.444]),
               np.array([ 1.476, -0.151, -1.735]),np.array([2.5, 0.45,  0.25, -1.2]),
               np.array([0.8, -0.9 , 0.5, -1.7]),
              np.array([1.8, -2.1 , -0.5, -1.7]), np.array([-1.24,  1.17, -1.04,  0.97])]



for sigmm in [0.25, 0.8, 1.2, 2.5]: # Set of true Frailty standard deviation 
    
    Res_simul_synth3=[]
    
    for beta_true in vec_beta_true[:]: # Loop over set fixed parameters


        tstart = time.time()


        print('Initial sigma:',sigmm)
        
#  ---------------------------------- PART 1 : Data genertating process ---------------------------
        

        # Simulate data in a longitudinal format
        np.random.seed(42)

        n_individuals = 10000
        obs_per_individual = np.random.randint(1, 6, size=n_individuals)  # Individuals may have between 1 and
#         6 observations (number of observation are assigned randomly per individual)
        n_obs = np.sum(obs_per_individual)  

        # True random effect standard deviation (frailty)
        sigma_u_true = sigmm

        # Generate random effects (frailty) per individual
        u = np.random.normal(0, sigma_u_true, size=n_individuals)

        # Generate fixed covariates
        
        if len(beta_true)==3:
            X1 = np.random.uniform(-2, 2, size=n_obs)  
            X2 = np.random.choice([0, 1], size=n_obs)
            
        elif len(beta_true)==4:
            X1 = np.random.uniform(-2, 2, size=n_obs)  
            X2 = np.random.choice([0, 1], size=n_obs) 
            X3 = np.random.gamma(1.2, .6, size=n_obs)

        # Assign ids to individuals to be able to track them
        individual_ids = np.repeat(np.arange(n_individuals), obs_per_individual)

        
        # Generate linear predictor and probabilities
        if len(beta_true)==3:
        
            eta = beta_true[0] + beta_true[1] * X1 + beta_true[2] * X2 + u[individual_ids] 
            p = 1 / (1 + np.exp(-eta))  
            
        elif len(beta_true)==4:
            eta = beta_true[0] + beta_true[1] * X1 + beta_true[2] * X2 + beta_true[3] * X3 + u[individual_ids] 
            p = 1 / (1 + np.exp(-eta))  

        



        # Generate binary responses
        y = np.random.binomial(1, p, size=n_obs)

        # Create DataFrame
        # 
        

        # Step 2: Prepare data for optimization
        if len(beta_true)==3:
            df = pd.DataFrame({'y': y, 'X1': X1, 'X2': X2, 'individual': individual_ids})
            X = sm.add_constant(df[['X1', 'X2']])  
            
        elif len(beta_true)==4:
            df = pd.DataFrame({'y': y, 'X1': X1, 'X2': X2, 'X3': X3, 'individual': individual_ids})
            X = sm.add_constant(df[['X1', 'X2', 'X3']])
            
            
        else:
            print("Issue for beta",beta_true)
            
        y = df['y'].values
        # Ensure individual IDs are mapped to sequential indices
        unique_ids, groups = np.unique(df['individual'].values, return_inverse=True)

        # Update n_individuals based on unique IDs
        n_individuals = len(unique_ids)




# -------------------------------- End of part 1 (ata genertating process) ---------------------------
        
        
# ------------------------------------ Part 2: Recovering the true parameters using GHQ/EM ------------------
        
#  Fitting a fixed effects LLink model to the data (i.e. X)

        model_logis = LogisticRegression(penalty="none")
        model_logis.fit(X.values,y)

        init_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
        init_model.fit(X, y)
        
#         Extracting the parameters estimates (i.e. fixed effects) of the model 
        init_params_better = init_model.coef_.flatten()

        # Extracting predicted probabilities on the train set to generate some statistics 
        predicted_probs = model_logis.predict_proba(X)[:,1]

        # Log-likelihood
        loglik_reduced = np.sum(y * np.log(predicted_probs + 1e-8) + (1 - y) * np.log(1 - predicted_probs + 1e-8))


        # Extract initial estimates for fixed effects
        beta_init = model_logis.coef_[0]


        # Step 5: Initialize sigma_u using residual standard deviation from the fixed effects model
        residuals = y - predicted_probs
        sigma_u_init = np.std(residuals)
        log_sigma_u_init = np.log(sigma_u_init + 1e-4)

        init_params_better = np.append(beta_init, log_sigma_u_init) 

        # ------------------------------ Optimisation time-independent frailty model ---------------------


        # RUnning GHQ optimisation
        result_full = minimize(loglik_glmm_ghq_vectorized, init_params_better, args=(X, y, groups, n_individuals), 
                                method="L-BFGS-B", options={'maxiter': 1000,  'verbose': 1},
                               jac='2-point')

        loglik_full = -result_full.fun  

        # Step 6: Compute LRT and RLRT for sigma_u
        num_fixed_params = len(result_full.x) - 1
        p_values_fixed = np.full(num_fixed_params + 1, np.nan)  

        # Extracting hessian
        hessian = sm.tools.numdiff.approx_hess(result_full.x, loglik_glmm_ghq_vectorized, args=(X, y, groups,
                                                                                               n_individuals))
        
        # Extracting standard error if scalar
        if np.linalg.matrix_rank(hessian) == hessian.shape[0]:
            se_glmm = np.sqrt(np.diag(inv(hessian)))[:-1]  
        else:
            se_glmm = np.sqrt(np.diag(pinv(hessian)))[:-1]

        # Computing pvalues
        if n_individuals > 100:
            p_values_fixed[:-1] = 2 * (1 - norm.cdf(np.abs(result_full.x[:-1] / se_glmm)))  # Wald test
        else:
            p_values_fixed[:-1] = np.nan  # Placeholder for small samples


        # Displaying some results ()
        dataframe_synth = pd.DataFrame({
            'Parameter': list(df.columns[1:]) + ['Sigma_u'],
            'Estimate (LLink_with_GHQ)': np.append(result_full.x[:-1], np.exp(result_full.x[-1]))
        })
#         print(dataframe_synth,'\n')

        #  Running EM optimisation
        
        tp_beta_hat_EM_ori_singl, tp_sigma_u_hat_EM_ori_singl, tp_vec_ll_trace_EM_ori_singl, tp_res_ =\
        em_logistic_glmm_check(X.values, y, groups, n_quad=20, verbose=True)

        dataframe_synth_EM = pd.DataFrame({
            'Parameter': list(df.columns[1:]) + ['Sigma_u'],
            'Estimate (LLink_with_EM)': np.append(tp_beta_hat_EM_ori_singl, tp_sigma_u_hat_EM_ori_singl)
        })
    
        
        
#         print(dataframe_synth_EM,'\n')

        dataframe_synth_f = pd.DataFrame([np.append(beta_true,sigmm),
                                          dataframe_synth['Estimate (LLink_with_GHQ)'],
                                          dataframe_synth_EM['Estimate (LLink_with_EM)']]).T

        dataframe_synth_f.columns = ['True parameters','Estimates GHQ', 'Estimate EM']

        print('Final Parameter estimates GHQ vs EM (i.e., for round of set of true parameters):\n')
        
        print(dataframe_synth_f,'\n\n\n')
    
        Res_simul_synth3.append([dataframe_synth_f, result_full, tp_res_])

        tstop = time.time()

        print('Time taken', (tstop-tstart)/60., 'minutes\n\n\n')
        
    B_Res_simul_synth3.append(Res_simul_synth3)

Initial sigma: 0.25


/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/3121810126.py:139: OptimizeWarning: Unknown solver options: verbose
  result_full = minimize(loglik_glmm_ghq_vectorized, init_params_better, args=(X, y, groups, n_individuals),
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/2261307601.py:79: OptimizeWarning: Unknown solver options: verbose
  res = minimize(


Iter 0: sigma_u = 0.6405, loglik = -24091.2405
Iter 1: sigma_u = 0.5106, loglik = -27968.6120
Iter 2: sigma_u = 0.4081, loglik = -33878.2678
Iter 3: sigma_u = 0.3278, loglik = -42458.4074
Iter 4: sigma_u = 0.2657, loglik = -53702.4883
Iter 5: sigma_u = 0.2190, loglik = -66086.0503
Iter 6: sigma_u = 0.1848, loglik = -77271.0626
Iter 7: sigma_u = 0.1602, loglik = -85886.3326
Iter 8: sigma_u = 0.1423, loglik = -91969.0863
Iter 9: sigma_u = 0.1290, loglik = -96161.1492
Iter 10: sigma_u = 0.1189, loglik = -99084.8627
Iter 11: sigma_u = 0.1110, loglik = -101176.4496
Iter 12: sigma_u = 0.1047, loglik = -102717.4256
Iter 13: sigma_u = 0.0996, loglik = -103884.0169
Iter 14: sigma_u = 0.0954, loglik = -104788.8208
Iter 15: sigma_u = 0.0919, loglik = -105505.5209
Iter 16: sigma_u = 0.0889, loglik = -106083.6375
Iter 17: sigma_u = 0.0864, loglik = -106557.2567
Iter 18: sigma_u = 0.0842, loglik = -106950.4936
Iter 19: sigma_u = 0.0823, loglik = -107281.1429
Iter 20: sigma_u = 0.0806, loglik = -1075

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/3121810126.py:139: OptimizeWarning: Unknown solver options: verbose
  result_full = minimize(loglik_glmm_ghq_vectorized, init_params_better, args=(X, y, groups, n_individuals),
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/2261307601.py:79: OptimizeWarning: Unknown solver options: verbose
  res = minimize(


Iter 0: sigma_u = 0.9962, loglik = -13205.1841
Iter 1: sigma_u = 0.8433, loglik = -14473.0465
Iter 2: sigma_u = 0.7131, loglik = -16357.5773
Iter 3: sigma_u = 0.6032, loglik = -18947.7470
Iter 4: sigma_u = 0.5107, loglik = -22480.5329
Iter 5: sigma_u = 0.4331, loglik = -27237.0999
Iter 6: sigma_u = 0.3683, loglik = -33478.2192
Iter 7: sigma_u = 0.3145, loglik = -41251.6431
Iter 8: sigma_u = 0.2706, loglik = -50103.6407
Iter 9: sigma_u = 0.2352, loglik = -59081.2511
Iter 10: sigma_u = 0.2071, loglik = -67219.1726
Iter 11: sigma_u = 0.1848, loglik = -74003.2714
Iter 12: sigma_u = 0.1670, loglik = -79394.7274
Iter 13: sigma_u = 0.1525, loglik = -83595.7500
Iter 14: sigma_u = 0.1407, loglik = -86863.6691
Iter 15: sigma_u = 0.1309, loglik = -89426.1501
Iter 16: sigma_u = 0.1226, loglik = -91459.9992
Iter 17: sigma_u = 0.1155, loglik = -93096.0639
Iter 18: sigma_u = 0.1094, loglik = -94428.9435
Iter 19: sigma_u = 0.1041, loglik = -95531.4434
Iter 20: sigma_u = 0.0994, loglik = -96454.2994
It

...

Iter 0: sigma_u = 1.1977, loglik = -22024.5954
Iter 1: sigma_u = 1.7037, loglik = -17942.4495
Iter 2: sigma_u = 2.4186, loglik = -15893.2071
Iter 3: sigma_u = 3.4374, loglik = -14823.8873
Iter 4: sigma_u = 4.3826, loglik = -13849.4648
Iter 5: sigma_u = 4.8089, loglik = -13148.3840
Iter 6: sigma_u = 4.9482, loglik = -12916.7695
Iter 7: sigma_u = 4.9891, loglik = -12850.8232
Iter 8: sigma_u = 5.0007, loglik = -12832.3488
Iter 9: sigma_u = 5.0042, loglik = -12827.2012
Iter 10: sigma_u = 5.0048, loglik = -12825.6593
Iter 11: sigma_u = 5.0050, loglik = -12825.3728
Iter 12: sigma_u = 5.0052, loglik = -12825.2997
Iter 13: sigma_u = 5.0051, loglik = -12825.2244
Iter 14: sigma_u = 5.0052, loglik = -12825.2898
Iter 15: sigma_u = 5.0051, loglik = -12825.2287
Iter 16: sigma_u = 5.0052, loglik = -12825.2746
Iter 17: sigma_u = 5.0053, loglik = -12825.2236
Iter 18: sigma_u = 5.0052, loglik = -12825.1877
Iter 19: sigma_u = 5.0053, loglik = -12825.2222
Iter 20: sigma_u = 5.0054, loglik = -12825.1938
It

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/3121810126.py:139: OptimizeWarning: Unknown solver options: verbose
  result_full = minimize(loglik_glmm_ghq_vectorized, init_params_better, args=(X, y, groups, n_individuals),
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/2261307601.py:79: OptimizeWarning: Unknown solver options: verbose
  res = minimize(


Iter 0: sigma_u = 2.0317, loglik = -14471.5337
Iter 1: sigma_u = 2.7869, loglik = -13190.4788
Iter 2: sigma_u = 3.7505, loglik = -12505.2618
Iter 3: sigma_u = 4.5391, loglik = -11855.9586
Iter 4: sigma_u = 4.9519, loglik = -11372.4632
Iter 5: sigma_u = 5.1267, loglik = -11170.2154
Iter 6: sigma_u = 5.1939, loglik = -11094.3930
Iter 7: sigma_u = 5.2187, loglik = -11066.6458
Iter 8: sigma_u = 5.2278, loglik = -11056.5747
Iter 9: sigma_u = 5.2311, loglik = -11052.9313
Iter 10: sigma_u = 5.2318, loglik = -11051.6153
Iter 11: sigma_u = 5.2323, loglik = -11051.3037
Iter 12: sigma_u = 5.2325, loglik = -11051.1078
Iter 13: sigma_u = 5.2324, loglik = -11051.0319
Iter 14: sigma_u = 5.2326, loglik = -11051.0560
Iter 15: sigma_u = 5.2327, loglik = -11051.0034
Iter 16: sigma_u = 5.2326, loglik = -11050.9405
Iter 17: sigma_u = 5.2327, loglik = -11050.9799
Iter 18: sigma_u = 5.2327, loglik = -11050.9402
Iter 19: sigma_u = 5.2327, loglik = -11050.9707
Iter 20: sigma_u = 5.2328, loglik = -11050.9409
It

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/3121810126.py:139: OptimizeWarning: Unknown solver options: verbose
  result_full = minimize(loglik_glmm_ghq_vectorized, init_params_better, args=(X, y, groups, n_individuals),
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/2261307601.py:79: OptimizeWarning: Unknown solver options: verbose
  res = minimize(


Iter 0: sigma_u = 1.2183, loglik = -20935.1377
Iter 1: sigma_u = 1.7206, loglik = -17141.3292
Iter 2: sigma_u = 2.4220, loglik = -15212.9857
Iter 3: sigma_u = 3.3878, loglik = -14213.7177
Iter 4: sigma_u = 4.3228, loglik = -13443.5205
Iter 5: sigma_u = 4.7975, loglik = -12752.8224
Iter 6: sigma_u = 4.9740, loglik = -12476.3207
Iter 7: sigma_u = 5.0326, loglik = -12385.1915
Iter 8: sigma_u = 5.0514, loglik = -12355.0518
Iter 9: sigma_u = 5.0574, loglik = -12344.6670
Iter 10: sigma_u = 5.0594, loglik = -12340.8489
Iter 11: sigma_u = 5.0600, loglik = -12339.3191
Iter 12: sigma_u = 5.0603, loglik = -12338.6491
Iter 13: sigma_u = 5.0604, loglik = -12338.3143
Iter 14: sigma_u = 5.0603, loglik = -12338.1531
Iter 15: sigma_u = 5.0604, loglik = -12338.0977
Iter 16: sigma_u = 5.0605, loglik = -12338.0452
Iter 17: sigma_u = 5.0605, loglik = -12337.9666
Iter 18: sigma_u = 5.0606, loglik = -12337.9681
Iter 19: sigma_u = 5.0605, loglik = -12337.9199
Iter 20: sigma_u = 5.0605, loglik = -12337.9676
It

In [6]:
# Saving the rsults of each simulation and benchmark above to a list

try:
    vec_update_tp_list_dat
except:
    vec_update_tp_list_dat = []

for j in range(len(B_Res_simul_synth3)):

    update_tp_list_dat = []
    tp_list_dat =[B_Res_simul_synth3[j][i][0] for i in range(len(B_Res_simul_synth3[j]))]

    for dat in tp_list_dat:

        if dat.shape[0] == 4:
            dat.index = ['beta_0', 'beta_1', 'beta_2', 'sigma_u']
        else:
            dat.index = ['beta_0', 'beta_1', 'beta_2', 'beta_3', 'sigma_u']

        update_tp_list_dat.append(dat)
        
    vec_update_tp_list_dat.append(update_tp_list_dat)

In [7]:
'kokokaka'
vec_update_tp_list_dat[1]

[         True parameters  Estimates GHQ  Estimate EM
 beta_0               0.5       0.518098     0.518666
 beta_1               0.5       0.495437     0.495259
 beta_2              -1.2      -1.216457    -1.216541
 sigma_u              0.8       0.772839     0.213941,
          True parameters  Estimates GHQ  Estimate EM
 beta_0               1.5       1.464638     1.469740
 beta_1              -0.8      -0.822129    -0.823051
 beta_2               1.3       1.339157     1.341286
 sigma_u              0.8       0.788510     0.233045,
          True parameters  Estimates GHQ  Estimate EM
 beta_0            -1.438      -1.407220    -1.407471
 beta_1             4.847       4.755697     4.755374
 beta_2            -0.444      -0.411510    -0.410933
 sigma_u            0.800       0.842100     0.259174,
          True parameters  Estimates GHQ  Estimate EM
 beta_0             1.476       1.468188     1.467953
 beta_1            -0.151      -0.162452    -0.162284
 beta_2            -1.735

#### Transforming the list above into a table (This is TABLE 2.12 in my Thesis)

In [12]:
final_dat_compar_EM_GHQ =\
pd.concat([pd.concat(vec_update_tp_list_dat[0]),
           pd.concat(vec_update_tp_list_dat[1]),\
pd.concat(vec_update_tp_list_dat[2]), pd.concat(vec_update_tp_list_dat[3])], axis=1)


final_dat_compar_EM_GHQ.columns = ['TP', 'Est. GHQ', 'Est. EM', 'TP', 'Est. GHQ', 'Est. EM',
                                   'TP', 'Est. GHQ', 'Est. EM','TP', 'Est. GHQ', 'Est. EM']
final_dat_compar_EM_GHQ.index = ['$beta_0$', '$beta_1$', '$beta_2$', '$sigma_u$', '$beta_0$',
                                 '$beta_1$', '$beta_2$',
       '$sigma_u$', '$beta_0$', '$beta_1$', '$beta_2$', '$sigma_u$', '$beta_0$', '$beta_1$',
       '$beta_2$', '$sigma_u$', '$beta_0$', '$beta_1$', '$beta_2$', '$beta_3$', '$sigma_u$',
       '$beta_0$', '$beta_1$', '$beta_2$', '$beta_3$', '$sigma_u$', '$beta_0$', '$beta_1$',
       '$beta_2$', '$beta_3$', '$sigma_u$', '$beta_0$', '$beta_1$', '$beta_2$', '$beta_3$',
       '$sigma_u$']

print(final_dat_compar_EM_GHQ.to_latex(float_format='%.3f'))

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & TP & Est. GHQ & Est. EM & TP & Est. GHQ & Est. EM & TP & Est. GHQ & Est. EM & TP & Est. GHQ & Est. EM \\
\midrule
$beta_0$ & 0.500 & 0.484 & 0.484 & 0.500 & 0.518 & 0.519 & 0.500 & 0.523 & 0.519 & 0.500 & 0.520 & 0.499 \\
$beta_1$ & 0.500 & 0.480 & 0.480 & 0.500 & 0.495 & 0.495 & 0.500 & 0.490 & 0.482 & 0.500 & 0.477 & 0.441 \\
$beta_2$ & -1.200 & -1.200 & -1.200 & -1.200 & -1.216 & -1.217 & -1.200 & -1.216 & -1.203 & -1.200 & -1.178 & -1.105 \\
$sigma_u$ & 0.250 & 0.284 & 0.066 & 0.800 & 0.773 & 0.214 & 1.200 & 1.175 & 2.837 & 2.500 & 2.418 & 4.800 \\
$beta_0$ & 1.500 & 1.446 & 1.449 & 1.500 & 1.465 & 1.470 & 1.500 & 1.506 & 1.511 & 1.500 & 1.527 & 1.146 \\
$beta_1$ & -0.800 & -0.816 & -0.817 & -0.800 & -0.822 & -0.823 & -0.800 & -0.822 & -0.802 & -0.800 & -0.775 & -0.635 \\
$beta_2$ & 1.300 & 1.400 & 1.402 & 1.300 & 1.339 & 1.341 & 1.300 & 1.325 & 1.292 & 1.300 & 1.280 & 1.045 \\
$sigma_u$ & 0.250 & 0.067 & 0.028 & 0.800 & 0.789 & 0.233 & 1.

# 2. Other functions in the Thesis 

- Due to a NDA with the company that provided the data. We are not allowed to provide but these functions can be easily extended to the simulated data above (constructed in the same format as the original data).
- **All numerical results provided below are to assist in testing the functinality of the codes, and make exploration easier and more intuitive; therefore they are not part of the thesis.**

- Below, we fix the simulation data to be the last one generated in the simulation study

- We need to add the unique event time per individual as well be able to track, therefore we complete the simulation data as follows:

In [14]:
#  Contenating corresponding individuals and response variable
tp_dat_X = pd.concat([pd.DataFrame(individual_ids), X, pd.DataFrame(y)],axis=1)

tp_dat_X.columns = ['sample_id', 'cons', 'X1', 'X2', 'X3','y']

# Adding unqiue time per event per individual 
tp_dat_X['times'] = tp_dat_X.groupby(['sample_id'])['cons'].cumcount()+1



In [15]:
tp_dat_X

,sample_id,cons,X1,X2,X3,y,times
0,0,1.0,-1.853012,1,0.862900,0,1
1,0,1.0,-0.958259,1,0.448140,0,2
2,0,1.0,-1.147509,0,0.131308,0,3
3,0,1.0,1.469742,1,0.244961,1,4
4,1,1.0,0.395480,1,0.722274,1,1
...,...,...,...,...,...,...,...
29898,9998,1.0,-0.692278,1,0.533335,0,3
29899,9998,1.0,-1.740829,0,1.259163,0,4
29900,9999,1.0,0.879946,0,0.548936,1,1
29901,9999,1.0,1.958406,0,0.318894,1,2


### 2.1.  Implementation of the objective function (negative marginal log-likelihood) using GHQ integration: Case of time-dependent linear frailties
- see section 2.5.2.4

In [16]:
def loglik_glmm_2d_ghq_vectorized(params, X, y, groups, n_individuals, times, n_quad=10):

    beta = params[:-2]  # Fixed effects
    log_sigma_a = params[-2]  # Log of sqrt(variance) for slope frailty
    log_sigma_b = params[-1]  # Log of sqrt(variance) for intercept frailty

    sigma_a = np.exp(log_sigma_a)
    sigma_b = np.exp(log_sigma_b)

    n_obs = X.shape[0]

    # Gauss-Hermite quadrature nodes and weights
    ghq_points_a, ghq_weights_a = hermgauss(n_quad)
    ghq_points_b, ghq_weights_b = hermgauss(n_quad)

    ghq_weights_a /= np.sqrt(np.pi)
    ghq_weights_b /= np.sqrt(np.pi)

    # Rescale nodes
    a_nodes = ghq_points_a * np.sqrt(2) * sigma_a
    b_nodes = ghq_points_b * np.sqrt(2) * sigma_b

    # Fixed effects
    eta_fixed = np.asarray(X @ beta)  # shape (n_obs,)

    # Expand a_nodes and b_nodes into full 2D grid
    A, B = np.meshgrid(a_nodes, b_nodes, indexing='ij')  # both (n_quad, n_quad)

    # Expand to observations: broadcasting
    A_obs = A[None, :, :]  # shape (1, n_quad, n_quad)
    B_obs = B[None, :, :]  # shape (1, n_quad, n_quad)
    times_obs = times[:, None, None]  # (n_obs, 1, 1)
    eta_fixed_obs = eta_fixed[:, None, None]  # (n_obs, 1, 1)

    # Compute full eta (linear predictor)
    eta_random = A_obs * times_obs + B_obs  # shape (n_obs, n_quad, n_quad)
    eta_all = eta_fixed_obs + eta_random

    # Logistic link
    p_all = 1 / (1 + np.exp(-np.clip(eta_all, -30, 30)))

    # Log-likelihoods per observation and quadrature pair
    y_obs = y[:, None, None]
    log_likelihoods = y_obs * np.log(p_all + 1e-8) + (1 - y_obs) * np.log(1 - p_all + 1e-8)  # (n_obs, n_quad, n_quad)

    # Now sum over observations by groups (individuals)
    loglik_per_observation = np.zeros((n_individuals, n_quad, n_quad))
    np.add.at(loglik_per_observation, groups, log_likelihoods)  # (n_individuals, n_quad, n_quad)

    # Now do 2D weighted logsumexp per individual
    weights_2d = np.outer(ghq_weights_a, ghq_weights_b)  # (n_quad, n_quad)

    loglik_per_individual = logsumexp(
        loglik_per_observation.reshape(n_individuals, -1),
        b=weights_2d.flatten(),
        axis=1
    ) + np.log(1/np.pi)  # ⚡ Remember 1/pi correction

    total_loglik = np.sum(loglik_per_individual)

    return -total_loglik  # Negative log-likelihood for minimization




#### 2.2.1 Example use case (Linear-time dependent frailties)

In [17]:
# Selecting only the covariates

X = copy.deepcopy(tp_dat_X.iloc[:,1:-2])

print('shape of X', X.shape)
print('shape of X', X.columns,'\n')


shape of X (29903, 4)
shape of X Index(['cons', 'X1', 'X2', 'X3'], dtype='object') 



In [19]:
# Extracting times
times = tp_dat_X['times'].values

In [20]:
# ------------------------------------------------

# Ensure individual IDs are mapped to sequential indices
unique_ids, groups = np.unique(tp_dat_X['sample_id'].values, return_inverse=True)

# Update n_individuals based on unique IDs
n_individuals = len(unique_ids)

# ------------------------------------------------
# Making a copy 
groups_encoded = copy.deepcopy(groups)


# ---------------------------------------
model_logis_td = LogisticRegression(penalty="none")
model_logis_td.fit(X.values,y)

# Predicting probabilities on the train set to generate some statistics 
predicted_probs_td = model_logis_td.predict_proba(X)[:,1]

# Log-likelihood
loglik_reduced_td = np.sum(y * np.log(predicted_probs_td + 1e-8) +\
                           (1 - y) * np.log(1 - predicted_probs_td + 1e-8))


# Extract initial estimates for fixed effects
beta_init_td = model_logis_td.coef_[0]

# Residuals for starting point
residuals_td = y - predicted_probs_td
sigma_u_init_td = np.std(residuals_td)
log_sigma_u_init_td = np.log(sigma_u_init_td + 1e-4)


# --------------------------------------------

log_sigma_a_init = log_sigma_u_init_td
log_sigma_b_init = log_sigma_u_init_td


# Build full initial parameter vector
init_params_better = np.concatenate([beta_init_td, [log_sigma_a_init, log_sigma_b_init]])

# Fit the full model

result_line = minimize(loglik_glmm_2d_ghq_vectorized, init_params_better,
                       args=(X, y, groups_encoded, n_individuals, times), 
                        method="L-BFGS-B", options={'maxiter': 1000, 'disp': True}, jac='2-point')


loglik_line = -result_line.fun  # Extract log-likelihood from full model

# Compute p-values for fixed effects
num_fixed_params = len(result_line.x) - 2  # Now two variance components

p_values_fixed = np.full(num_fixed_params + 2, np.nan)  # Include variances

# --------------------------------------------------------------------------------------


if n_individuals > 100:
    se_glmm = np.sqrt(np.diag(
        pinv(sm.tools.numdiff.approx_hess(result_line.x, loglik_glmm_2d_ghq_vectorized,
                                          args=(X, y, groups_encoded, n_individuals, times)))
    ))[:-2]
    p_values_fixed[:-2] = 2 * (1 - norm.cdf(np.abs(result_line.x[:-2] / se_glmm)))
else:
    pass
# -------------------------------------------------------------------------------


#  Display results
tp_dataframe_line = pd.DataFrame({
    'Parameter': list(X.columns) + ['Sigma_a (slope)', 'Sigma_b (intercept)'],
    'True parameters': np.concatenate([beta_true, [np.nan, sigmm]]),
    'Parameter Estimate (GHQ)': np.concatenate([result_line.x[:-2], np.exp(result_line.x[-2:])]),
    'p-value (Fixed effects)': p_values_fixed
})

tp_dataframe_line

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/1584080852.py:46: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result_line = minimize(loglik_glmm_2d_ghq_vectorized, init_params_better,


,Parameter,True parameters,Parameter Estimate (GHQ),p-value (Fixed effects)
0,cons,-1.24,-1.148291,0.0
1,X1,1.17,1.159556,0.0
2,X2,-1.04,-1.118424,0.0
3,X3,0.97,0.953097,0.0
4,Sigma_a (slope),NaN,0.127494,NaN
5,Sigma_b (intercept),2.50,2.458066,NaN


### 2.3. Implementation of the objective function (negative marginal log-likelihood) using GHQ integration: Case of time-dependent piecewise frailties
- See section 2.5.2.5

In [21]:
def loglik_glmm_piecewise_frailty(params, X, y, groups, n_individuals, times, tau=[3, 5, 10], n_quad=20):
    """
    Piecewise time-dependent frailty model using Gauss-Hermite quadrature.
    """
    n_obs = len(y)
    n_segments = len(tau)

    # Extract parameters
    beta = params[:-n_segments]
    log_sigmas = params[-n_segments:]
    sigmas = np.exp(log_sigmas)  # Convert to positive scale

    # Gauss-Hermite nodes/weights (shared)
    gh_x, gh_w = hermgauss(n_quad)
    gh_w = gh_w / np.sqrt(np.pi)

    # Precompute fixed part of eta
    eta_fixed = X @ beta
    loglik_total = 0

    for k in range(n_segments): # For each time interval tau_k \in {tau_1, tau_2, tau_3}
        

        # Identifying relevant intervals
        t_low = 0 if k == 0 else tau[k - 1]
        t_high = tau[k]

        # Selecting observations in this time segment
        idx_k = np.where((times > t_low) & (times <= t_high))[0]
        if len(idx_k) == 0:
            continue  # No data in this segment

        X_k = X.values[idx_k] if isinstance(X, pd.DataFrame) else X[idx_k]
        y_k = y[idx_k]
        
        eta_k = eta_fixed[idx_k]
        eta_k = np.asarray(eta_k)
        
        groups_k = groups[idx_k]

        # Rescaling nodes for this sigma
        u_k = np.sqrt(2) * sigmas[k] * gh_x
        eta_all = eta_k[:, None] + u_k[None, :]  # shape (n_obs_k, n_quad)
        p_all = 1 / (1 + np.exp(-np.clip(eta_all, -30, 30)))

        # Computing log-likelihoods
        logliks = y_k[:, None] * np.log(p_all + 1e-8) + (1 - y_k[:, None]) * np.log(1 - p_all + 1e-8)

        
#         print(f"Segment {k+1} | sigma: {sigmas[k]:.4f} | u_k range: {u_k.min():.2f} to {u_k.max():.2f}")

        # Groupwise summation
        loglik_per_obs = np.zeros((n_individuals, n_quad))
        np.add.at(loglik_per_obs, groups_k, logliks)

        # Integrate over quadrature points
        loglik_per_indiv = logsumexp(loglik_per_obs, b=gh_w, axis=1)
        loglik_total += np.sum(loglik_per_indiv)

    return -loglik_total  # for minimization


#### 2.3.1 Example use case (Linear-time dependent frailties)

In [22]:

# ------------------------------------------------

# Ensure individual IDs are mapped to sequential indices
unique_ids, groups = np.unique(tp_dat_X['sample_id'].values, return_inverse=True)

# Update n_individuals based on unique IDs
n_individuals = len(unique_ids)

groups_encoded = copy.deepcopy(groups)


#  Reduced model (no frailty)
logit_model = sm.Logit(y, X)
logit_result = logit_model.fit(disp=0)
loglik_reduced = logit_result.llf

#  Initialising frailty std devs for 3 segments
predicted_probs = logit_result.predict(X)
residuals = y - predicted_probs
sigma_resid = np.std(residuals)

#  Setting frailties initial values from residuals standard deviation
log_sigma_1 = np.log(sigma_resid + 1e-4)
log_sigma_2 = np.log(sigma_resid + 1e-4)
log_sigma_3 = np.log(sigma_resid + 1e-4)

# Initial parameter vector
beta_init = logit_result.params.values
init_params_piecewise = np.concatenate([beta_init, [log_sigma_1, log_sigma_2, log_sigma_3]])
# print('len(beta_init):',len(beta_init))

#  Fit the full piecewise model
result_piecewise = minimize(loglik_glmm_piecewise_frailty, init_params_piecewise,
                       args=(X, y, groups_encoded, n_individuals, times), 
                        method="L-BFGS-B", options={'maxiter': 1000, 'disp': True}, jac='2-point')




loglik_piecewise = -result_piecewise.fun
num_fixed_params = len(result_piecewise.x) - 3

# P-values for fixed effects
p_values_fixed = np.full(num_fixed_params + 3, np.nan)

if n_individuals > 100:
    se_glmm = np.sqrt(np.diag(
        pinv(sm.tools.numdiff.approx_hess(result_piecewise.x, loglik_glmm_piecewise_frailty,
                                          args=(X, y, groups_encoded, n_individuals, times)))
    ))[:-3]
    p_values_fixed[:-3] = 2 * (1 - norm.cdf(np.abs(result_piecewise.x[:-3] / se_glmm)))
else:
    pass


# Results in. a table
tp_dataframe_piecewise = pd.DataFrame({
    'Parameter': list(X.columns) + ['Sigma_1 (early)', 'Sigma_2 (mid)', 'Sigma_3 (late)'],
    'True parameters': np.concatenate([beta_true, [sigmm, sigmm, sigmm]]),
    'Parameter Estimate (GHQ)': np.concatenate([result_piecewise.x[:-3], np.exp(result_piecewise.x[-3:])]),
    'p-value (Fixed)': p_values_fixed
})

tp_dataframe_piecewise

/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/996050598.py:33: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result_piecewise = minimize(loglik_glmm_piecewise_frailty, init_params_piecewise,


,Parameter,True parameters,Parameter Estimate (GHQ),p-value (Fixed)
0,cons,-1.24,-1.142947,0.0
1,X1,1.17,1.142291,0.0
2,X2,-1.04,-1.099285,0.0
3,X3,0.97,0.953337,0.0
4,Sigma_1 (early),2.50,2.437360,NaN
5,Sigma_2 (mid),2.50,2.563610,NaN
6,Sigma_3 (late),2.50,0.451681,NaN


In [23]:
'poplo'

'poplo'

# 3. Parametric bootsrap LRT (PB-LR) functions

- The parametric bootstrap LRT approach helps to compute the statistical significance (p-value) of the frailty variance (s). See section 2.5.5 of my thesis.

### 3.1 Implementation (T1) : Fixed-effects vs. time-independent frailty (random intercept)

In [25]:
# Simulate under H0: σ² = 0 (no frailty)
# # -----------------------------

# Function to simulate response variables y from fixed effects model
def simulate_null_data(X, beta, rng=None):
    eta = X @ beta
    eta = np.clip(eta, -30, 30)
#     eta = np.clip(eta, -10, 10)
    p = 1 / (1 + np.exp(-eta))
    if rng is None:
        rng = np.random.default_rng()
    return rng.binomial(1, p, size=len(p))

#  Parametric bootstrap LRT function (implemented to run in parallel over the bootstrap replicates)
def wrap_parametric_bootstrap_rlrt_with_tracking(X, groups, beta_null, n_individuals,
                                                 compute_full_loglik_fn,
                                                 B: int = 100, n_quad: int = 20,
                                                 n_jobs: int = -1, seed: int = 42):
    def single_bootstrap_run(b):
#         Setting seed at each bootstrap for reproducibility
        rng = np.random.default_rng(seed + b)

#         Simulating the response variables from the reduced model (model with no frailty in this case)
        y_sim = simulate_null_data(X, beta_null, rng=rng)

#         Fitting Simple LLink model
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model_red = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
            model_red.fit(X, y_sim)
            if model_red.n_iter_[0] >= model_red.max_iter:
                return (None, 'no_converge_reduced', None, None)

        beta_red_sim = model_red.coef_.flatten()
        model_logis_red_paraams = model_red.coef_[0]
        predicted_probs = model_red.predict_proba(X)[:, 1]
        
#         Extracting reduced model loglikelihood
        loglik_red_sim = np.sum(y_sim * np.log(predicted_probs + 1e-8) +
                                (1 - y_sim) * np.log(1 - predicted_probs + 1e-8))
        

        try:
            def neg_loglik_full(params):
                return compute_full_loglik_fn(params, X, y_sim, groups, n_individuals, n_quad=n_quad)
            
#             Extracting residuals from fixed effects model to setup initial value of frailty variance
            residuals = y_sim - predicted_probs
            sigma_u_init = np.std(residuals) # Initial value for time-independent frailties
                 
#             Setting initial paramters
            init_params = np.concatenate([np.array(model_logis_red_paraams), np.log([sigma_u_init])])

        except:
            return (None, 'no_converge_reduced', None, None)

        res = minimize(neg_loglik_full, init_params, method="L-BFGS-B", options={'maxiter': 1000})
        if not res.success:
            return (None, 'no_converge_reduced', None, None)

        loglik_full_sim = -res.fun
        stat = 2 * (loglik_full_sim - loglik_red_sim)
        return (stat, None, loglik_full_sim, loglik_red_sim)

#     Running in parallel over B boostrap replicates 
    results = Parallel(n_jobs=n_jobs)(delayed(single_bootstrap_run)(b) for b in range(B))

#     Savinf the some relevant results and relevant statistics into lists
    valid_stats = [r[0] for r in results if r[0] is not None] # Here is the list of parametric bootstrap LRT statistics
    failures = [r[1] for r in results if r[0] is None] # Failed bootstrap (i.e. no convergence can be identified here)
    loglik_full_list = [r[2] for r in results if r[0] is not None]# List of parametric bootstrap MLE values for the full model at corresponding bootstrap resamples
    loglik_red_list = [r[3] for r in results if r[0] is not None] # List of parametric bootstrap MLE values for the reduced model at corresponding bootstrap resamples
    failure_counts = dict(Counter(failures))

    return np.array(valid_stats), failure_counts, results, loglik_full_list, loglik_red_list

#### 3.1.1. Example use case for the above function 

In [26]:
# Fixed effects model
model_logis_reduced = LogisticRegression(penalty="none") # Setting instance
model_logis_reduced.fit(X.values,y)# Fitting
model_logis_reduced_paraams = model_logis_reduced.coef_[0] # Extrating fixed effects vector of estimates

# Extracting predicted probabilities on same data 
predicted_probs_reduced_model = model_logis_reduced.predict_proba(X)[:,1]

loglik_reduced_model = np.sum(y * np.log(predicted_probs_reduced_model + 1e-8) +\
                           (1 - y) * np.log(1 - predicted_probs_reduced_model + 1e-8))


residuals = y - predicted_probs_reduced_model
sigma_u_init = np.std(residuals) # Initial value for time-independent frailties
log_sigma_u_init = np.log(sigma_u_init + 1e-4) # Making sure log of initial values does not blow up by adding a
# small epsilon
init_params_time_indep = np.append(model_logis_reduced_paraams, log_sigma_u_init)  # setting up vector of full
# initial parameters

    
# ---------------------- Full model (observed) ---------------------

result_full = minimize(loglik_glmm_ghq_vectorized, init_params_time_indep, args=(X, y, groups, n_individuals), 
                        method="L-BFGS-B", options={'maxiter': 1000,  'verbose': 1},
                       jac='2-point')

loglik_full_model_obs = -result_full.fun  


# ----------------- Parametric bootstrap LRT : Case of time-independent frailty ---------------------------
tstart = time.time()


B=100 # we set the number of bootstrap B to be low so that one can judge how long it may take to finish.
# The number can then be increased after this has been tested

ressss_0 =\
    wrap_parametric_bootstrap_rlrt_with_tracking(
    X=X,
    groups=groups,
    beta_null=model_logis_reduced_paraams,
    n_individuals=n_individuals,
    compute_full_loglik_fn=loglik_glmm_ghq_vectorized,
    B=B,
    n_quad=20,
    n_jobs=-1,
    seed=42
)

tstop = time.time()

print('Time taken', (tstop-tstart)/60., 'minutes')

# ----------------------------------------------------

# Compute log-likelihood from your actual full piecewise model

# Negating the full model the maximum likelihood estimate obtained original training set (i.e.  minus the
# minimize loglikelihod)
max_log_lik_full_time_indep_case = loglik_full_model_obs
max_log_lik_freduced_time_indep_case = loglik_reduced_model # loglik_reduced_model should be the loglikelihood
# estimate from the last simulation study in section 1.3 of this notebook

# Making sure the statistics (i.e. the LRT test statistics) is not negative
tp_lrt_observed_time_indep_case = max(2 * ( max_log_lik_full_time_indep_case - max_log_lik_freduced_time_indep_case), 0)
resss_boot_time_indep_case_LRT = np.where(ressss_0[0]<0,0,ressss_0[0])

tp_pval_rlr_time_indep_case = np.mean(resss_boot_time_indep_case_LRT >= tp_lrt_observed_time_indep_case)


print(f"RLRT p-value for frailty variance(s): {tp_pval_rlr_time_indep_case:.4f}")

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/4088397381.py:23: OptimizeWarning: Unknown solver options: verbose
  result_full = minimize(loglik_glmm_ghq_vectorized, init_params_time_indep, args=(X, y, groups, n_individuals),


Time taken 0.8973313848177592 minutes
RLRT p-value for frailty variance(s): 0.0000


**N.B. The PB-LRT remaining functions presented below can be explored in a similar procedures**

### 3.2 Implementation of (T2) : Linear time-dependent frailty: test of slope variance (conditional on intercept variance)

In [28]:
def wrap_parametric_bootstrap_rlrt_sigma_a_only(
    X, times, groups, n_individuals,
    compute_loglik_full,  # full model: uses both a_i and b_i
    compute_loglik_reduced,  # reduced model: uses only b_i
    beta_null, sigma_b_null,
    B=100, n_quad=20, n_jobs=-1, seed=42
):
    def simulate_from_reduced_model(rng):
        b_i = rng.normal(loc=0.0, scale=sigma_b_null, size=n_individuals)
        eta = (X @ beta_null) + b_i[groups]
        p = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        return rng.binomial(1, p)

    def single_bootstrap_run(b):
        rng = np.random.default_rng(seed + b)
#         try:
        y_sim = simulate_from_reduced_model(rng)

        # Fit reduced model (b_i only)
        def neg_loglik_red(params):
            return compute_loglik_reduced(params, X, y_sim, groups, n_individuals, n_quad=n_quad)

        init_red = np.concatenate([beta_null, [np.log(sigma_b_null)]])
        res_red = minimize(neg_loglik_red, init_red, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_red.success:
            return (None, 'no_converge_reduced', None, None)
        loglik_red = -res_red.fun
        beta_est_red = res_red.x[:-1]
        log_sigma_b_est = res_red.x[-1]
        
#         -----------------------------------------------
        
        tp_eta = X @ beta_null
        prob_red =  1 / (1 + np.exp(-np.clip(tp_eta, -30, 30)))
        
        tp_residuals = y_sim - prob_red
        tp_sigma_resid = np.std(tp_residuals)
        
#         -----------------------------------------------
        

        # Fit full model (a_i and b_i)
        def neg_loglik_full(params):
            return compute_loglik_full(params, X, y_sim, groups, n_individuals, times, n_quad=n_quad)

#         init_full = np.concatenate([beta_est_red, np.log([1e-1, sigma_b_null])])
        init_full = np.concatenate([beta_est_red, [np.log(1e-1), log_sigma_b_est]])
#         init_full = np.concatenate([beta_est_red, [np.log(tp_sigma_resid), log_sigma_b_est]])
        res_full = minimize(neg_loglik_full, init_full, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_full.success:
            return (None, 'no_converge_full', None, None)

        loglik_full = -res_full.fun
        stat = 2 * (loglik_full-loglik_red)
        return (stat, None, loglik_full, loglik_red)

#         except Exception:
#             return (None, 'exception', None, None)

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap_run)(b) for b in range(B)
    )

    valid_stats = [r[0] for r in results if r[0] is not None]
    failures = [r[1] for r in results if r[0] is None]
    loglik_full_list = [r[2] for r in results if r[0] is not None]
    loglik_red_list = [r[3] for r in results if r[0] is not None]
    failure_counts = dict(Counter(failures))

    return np.array(valid_stats), failure_counts, results, loglik_full_list, loglik_red_list


In [29]:
# Computing p-values for the time-independent case based for the time-independent case


model_logis_fix_effects = LogisticRegression(penalty="none")
model_logis_fix_effects.fit(X.values,y)
model_logis_params_fix_effects = model_logis_fix_effects.coef_[0]

# Predicting probabilities on the train set to generate some statistics 
predicted_probs_fix_effects = model_logis_fix_effects.predict_proba(X)[:,1]

loglik_reduced_fix_effects = np.sum(y * np.log(predicted_probs_fix_effects + 1e-8) +\
                           (1 - y) * np.log(1 - predicted_probs_fix_effects + 1e-8))


residuals = y - predicted_probs_fix_effects
sigma_u_init = np.std(residuals) # Initial value for time-independent frailties
log_sigma_u_init = np.log(sigma_u_init + 1e-4) # Making sure log of initial values does not blow up by adding a
# small epsilon
init_params_time_indep = np.append(model_logis_params_fix_effects, log_sigma_u_init)  # setting up vector of full
# initial parameters


result_reduced_time_indep = minimize(loglik_glmm_ghq_vectorized, init_params_time_indep, args=(X, y, groups,
                                                                            n_individuals), 
                        method="L-BFGS-B", options={'maxiter': 1000, 'disp': True}, jac='2-point')



tstart= time.time()

resss_a_case = wrap_parametric_bootstrap_rlrt_sigma_a_only(
    X, times, groups, n_individuals,
    loglik_glmm_2d_ghq_vectorized,  # full model: uses both a_i and b_i
    loglik_glmm_ghq_vectorized,  # reduced model: uses only b_i
    result_reduced_time_indep.x[:-1],
    np.exp(result_reduced_time_indep.x[-1]),
    B=25, n_quad=20, n_jobs=-1, seed=42)  # Try with a small B first to see how long it takes, the increase

tstop= time.time()

print('Time taken is',(tstop-tstart)/60.,'minutes')

# Negating the full model the maximum likelihood estimate obtained original training set (i.e.  minus the
# minimize loglikelihod)
max_log_lik_full_linear_case = loglik_line
max_log_lik_freduced_linear_case = -result_reduced_time_indep.fun

# Making sure the statistics (i.e. the LRT test statistics) is not negative
tp_lrt_observed_line_case = max(2 * ( max_log_lik_full_linear_case - max_log_lik_freduced_linear_case), 0)
resss_boot_line_case_LRT = np.where(resss_a_case[0]<0,0,resss_a_case[0])


# Computing bootstrap parametric LRT p-value
tp_pval_rlrt_case_line = np.mean(resss_boot_line_case_LRT >= tp_lrt_observed_line_case)


print(f"RLRT p-value for frailty variance(s): {tp_pval_rlrt_case_line:.4f}")

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_59340/3669965178.py:23: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result_reduced_time_indep = minimize(loglik_glmm_ghq_vectorized, init_params_time_indep, args=(X, y, groups,
/Users/ced001/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory 

Time taken is 5.83971730073293 minutes
RLRT p-value for frailty variance(s): 1.0000


### 3.3 Implementation of (T3) : Linear time-dependent frailty: test of intercept variance (conditional on slope variance)

In [36]:
def wrap_parametric_bootstrap_rlrt_sigma_b_only(
    X, times, groups, n_individuals,
    compute_loglik_full,  # full model: uses both a_i and b_i
    compute_loglik_reduced,  # reduced model: uses only b_i
    beta_null, sigma_a_null,
    B=100, n_quad=20, n_jobs=-1, seed=42
):
    def simulate_from_reduced_model(rng):
        a_i = rng.normal(loc=0.0, scale=sigma_a_null, size=n_individuals)
        eta = (X @ beta_null) + a_i[groups]*times
        p = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        return rng.binomial(1, p)

    def single_bootstrap_run(b):
        rng = np.random.default_rng(seed + b)
#         try:
        y_sim = simulate_from_reduced_model(rng)

        # Fit reduced model (b_i only)
        def neg_loglik_red(params):
            return compute_loglik_reduced(params, X, y_sim, groups, n_individuals, times, n_quad=n_quad)

        init_red = np.concatenate([beta_null, [np.log(sigma_a_null)]])
        res_red = minimize(neg_loglik_red, init_red, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_red.success:
            return (None, 'no_converge_reduced', None, None)
        loglik_red = -res_red.fun
        beta_est_red = res_red.x[:-1]
        log_sigma_a_est = res_red.x[-1]
        
#         -----------------------------------------------
        
        tp_eta = X @ beta_null
        prob_red =  1 / (1 + np.exp(-np.clip(tp_eta, -30, 30)))
        
        tp_residuals = y_sim - prob_red
        tp_sigma_resid = np.std(tp_residuals)
        
#         -----------------------------------------------
        

        # Fit full model (a_i and b_i)
        def neg_loglik_full(params):
            return compute_loglik_full(params, X, y_sim, groups, n_individuals, times, n_quad=n_quad)

#         init_full = np.concatenate([beta_est_red, np.log([sigma_a_null, 1e-1])])
        init_full = np.concatenate([beta_est_red, [log_sigma_a_est, np.log(1e-1)]])
#         init_full = np.concatenate([beta_est_red, [log_sigma_b_est, np.log(tp_sigma_resid)]])
        res_full = minimize(neg_loglik_full, init_full, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_full.success:
            return (None, 'no_converge_full', None, None)

        loglik_full = -res_full.fun
        stat = -2 * (loglik_red - loglik_full)
        return (stat, None, loglik_full, loglik_red)

#         except Exception:
#             return (None, 'exception', None, None)

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap_run)(b) for b in range(B)
    )

    valid_stats = [r[0] for r in results if r[0] is not None]
    failures = [r[1] for r in results if r[0] is None]
    loglik_full_list = [r[2] for r in results if r[0] is not None]
    loglik_red_list = [r[3] for r in results if r[0] is not None]
    failure_counts = dict(Counter(failures))

    return np.array(valid_stats), failure_counts, results, loglik_full_list, loglik_red_list


### 3.4 Implementation of (T4) : Piecewise time-dependent frailty - global test

In [37]:
def wrap_parametric_bootstrap_rlrt_piecewise_any(
    X, times, groups, n_individuals,
    compute_loglik_full,  # e.g., loglik_glmm_piecewise_frailty
    beta_null,
    B=100, n_quad=20, tau=None, n_jobs=-1, seed=42
):

    def simulate_from_null(rng):
        eta = X @ beta_null
        p = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        return rng.binomial(1, p)

    def single_bootstrap_run(b):
        rng = np.random.default_rng(seed + b)

        y_sim = simulate_from_null(rng)

        # Reduced model: fixed effects only
        model_red = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model_red.fit(X, y_sim)
            if model_red.n_iter_[0] >= model_red.max_iter:
                return (None, 'no_converge_reduced', None, None)
        beta_red_sim = model_red.coef_.flatten()
        prob_red = model_red.predict_proba(X)[:, 1]
        loglik_red_sim = np.sum(
            y_sim * np.log(prob_red + 1e-8) +
            (1 - y_sim) * np.log(1 - prob_red + 1e-8)
        )
        
        
        

        tp_residuals = y_sim - prob_red
        tp_sigma_resid = np.std(tp_residuals)

        # Full model: piecewise frailty
        def neg_loglik_full(params):
            return compute_loglik_full(params, X, y_sim, groups, n_individuals, times, tau=tau, n_quad=n_quad)

        n_segments = len(tau)
#         init_params = np.concatenate([beta_red_sim, np.log([tp_sigma_resid] * n_segments)])
        init_params = np.concatenate([beta_red_sim, np.log([1e-1] * n_segments)])
        res_full = minimize(neg_loglik_full, init_params, method="L-BFGS-B", options={'maxiter': 1000})
        if not res_full.success:
            return (None, 'no_converge_full', None, None)

        loglik_full_sim = -res_full.fun
        stat = -2 * (loglik_red_sim - loglik_full_sim)
        return (stat, None, loglik_full_sim, loglik_red_sim)



    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap_run)(b) for b in range(B)
    )

    valid_stats = [r[0] for r in results if r[0] is not None]
    failures = [r[1] for r in results if r[0] is None]
    loglik_full_list = [r[2] for r in results if r[0] is not None]
    loglik_red_list = [r[3] for r in results if r[0] is not None]
    failure_counts = dict(Counter(failures))

    return np.array(valid_stats), failure_counts, results, loglik_full_list, loglik_red_list


### 3.5 Implementation of (T5) : Piecewise time-dependent frailty: component-wise tests.

- The function **wrap_parametric_bootstrap_rlrt_piecewise_component** generate the PB-LRT statistics from the function **loglik_glmm_piecewise_frailty** (which should be used **as the full model** in this case) and the **reduced model** (in this case the function **loglik_glmm_piecewise_frailty_exclude_k**, which sets the k^th frailty variance to 0) 

In [38]:
def wrap_parametric_bootstrap_rlrt_piecewise_component(
    X, times, groups, n_individuals,
    compute_loglik_full,  # e.g., loglik_glmm_piecewise_frailty
    compute_loglik_reduced,  # e.g., loglik_glmm_piecewise_frailty_exclude_k
    beta_null,
    sigma_nulls_reduced,  # log sigmas with the tested component excluded
    component_index,  # index of the variance component being tested (0-based)
    B=100, n_quad=20, tau=None, n_jobs=-1, seed=42
):

    if tau is None:
        raise ValueError("tau must be provided.")

    def simulate_from_reduced_model(rng):
        n_segments = len(tau)
        sigmas = np.exp(np.array(sigma_nulls_reduced))
        u = np.zeros((n_individuals, n_segments))
        sigma_idx = 0
        for k in range(n_segments):
            if k == component_index:
                continue
            u[:, k] = rng.normal(loc=0.0, scale=sigmas[sigma_idx], size=n_individuals)
            sigma_idx += 1
        t_mask = [(times > (0 if k == 0 else tau[k - 1])) & (times <= tau[k]) for k in range(n_segments)]
        eta = X @ beta_null
        for k in range(n_segments):
            eta += u[groups, k] * t_mask[k]
        p = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        return rng.binomial(1, p)

    def single_bootstrap_run(b):
        rng = np.random.default_rng(seed + b)
        y_sim = simulate_from_reduced_model(rng)

        # Reduced model
        def neg_loglik_reduced(params):
            return compute_loglik_reduced(params, X, y_sim, groups, n_individuals,
                                          times, tau=tau, component_index=component_index, n_quad=n_quad)

        init_red = np.concatenate([beta_null, sigma_nulls_reduced])
        res_red = minimize(neg_loglik_reduced, init_red, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_red.success:
            return (None, 'no_converge_reduced', None, None)
        loglik_red = -res_red.fun
        beta_est = res_red.x[:len(beta_null)]
        

        # Full model
        n_segments = len(tau)
        sigma_idx = 0
        log_sigmas_full = []
        for i in range(n_segments):
            if i == component_index:
                log_sigmas_full.append(np.log(5e-1))  # inserting small init for omitted one
            else:
                log_sigmas_full.append(sigma_nulls_reduced[sigma_idx])
                sigma_idx += 1
        init_full = np.concatenate([beta_est, log_sigmas_full])

        def neg_loglik_full(params):
            return compute_loglik_full(params, X, y_sim, groups, n_individuals, times, tau=tau, n_quad=n_quad)

        res_full = minimize(neg_loglik_full, init_full, method="L-BFGS-B", options={"maxiter": 1000})
        if not res_full.success:
            return (None, 'no_converge_full', None, None)

        loglik_full = -res_full.fun
        stat = -2 * (loglik_red - loglik_full)
        return (stat, None, loglik_full, loglik_red)

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_bootstrap_run)(b) for b in range(B)
    )

    valid_stats = [r[0] for r in results if r[0] is not None]
    failures = [r[1] for r in results if r[0] is None]
    loglik_full_list = [r[2] for r in results if r[0] is not None]
    loglik_red_list = [r[3] for r in results if r[0] is not None]
    failure_counts = dict(Counter(failures))

    return np.array(valid_stats), failure_counts, results, loglik_full_list, loglik_red_list


In [39]:
def loglik_glmm_piecewise_frailty_exclude_k(
    params, X, y, groups, n_individuals, times, tau=[3, 5, 10],
    component_index=None, n_quad=20
):

    n_segments = len(tau)
    assert component_index is not None and 0 <= component_index < n_segments, "Invalid component_index"

    beta = params[:- (n_segments - 1)]
    log_sigmas = params[- (n_segments - 1):]
    sigmas = np.exp(log_sigmas)
    
#     print('len(beta):',len(beta))

    gh_x, gh_w = hermgauss(n_quad)
    gh_w = gh_w / np.sqrt(np.pi)

    eta_fixed = X @ beta
    loglik_total = 0
    sigma_idx = 0

    for k in range(n_segments):
        if k == component_index:
            continue  # Skip this segment (variance = 0)

        t_low = 0 if k == 0 else tau[k - 1]
        t_high = tau[k]
        idx_k = np.where((times > t_low) & (times <= t_high))[0]
        if len(idx_k) == 0:
            continue

        X_k = X.iloc[idx_k] if hasattr(X, "iloc") else X[idx_k]
        y_k = y[idx_k]
        eta_k = eta_fixed.iloc[idx_k] if hasattr(eta_fixed, "iloc") else eta_fixed[idx_k]
        eta_k = np.asarray(eta_k)
        groups_k = groups[idx_k]

        u_k = np.sqrt(2) * sigmas[sigma_idx] * gh_x
        sigma_idx += 1

        eta_all = eta_k[:, None] + u_k[None, :]
        p_all = 1 / (1 + np.exp(-np.clip(eta_all, -30, 30)))

        logliks = y_k[:, None] * np.log(p_all + 1e-8) + (1 - y_k[:, None]) * np.log(1 - p_all + 1e-8)

        loglik_per_obs = np.zeros((n_individuals, n_quad))
        np.add.at(loglik_per_obs, groups_k, logliks)
        loglik_per_indiv = logsumexp(loglik_per_obs, b=gh_w, axis=1)
        loglik_total += np.sum(loglik_per_indiv)

    return -loglik_total


# 4. Optimised Matthews Correlation Coefficient

### 4.1 The multistate classification functions are given as follows
- The functions have not been streamlined to automatically handle any number of states yet
- They have been written specifically for multistate classification when the number of states is 3 but can easily be extented:
    - This implies modifying ['Prob_land_state_1', 'Prob_land_state_2', 'Prob_land_state_3'] to an the exact  landing states one whishes to predict to 
    - For instance, for 4 states, one can set : ['Prob_land_state_1', 'Prob_land_state_2', 'Prob_land_state_3', 'Prob_land_state_4'] or whatever the columns of predicted probabilities in the input data to the function (i.e., Pred_data_0) have been named
    - Given an initial state h, The column "Prob_land_state_1" is the column of predicted probabilities for transition-type (h,1). "Prob_land_state_2" is the column of predicted probabilities for transition-type (h,2), and so on

In [4]:

# This function corresponds to the D&C methods for multitate classification
def vectorized_discrepancy_model(c_vec, Pred_data_0):
    
    # c_vec is the initial value of the vector of cut-off poins
    # Pred_data_0 is the training data containing appropriately named columns for the predicted transition 
    # probabilities 
    
    Pred_data = Pred_data_0.copy().reset_index(drop=True)

    # Extracting probability matrix and implementing discrepancy criteria
    probs = Pred_data[['Prob_land_state_1', 'Prob_land_state_2', 'Prob_land_state_3']].values
    adjusted_probs = probs - c_vec

    # prediction of next state
    pred_states = np.argmax(adjusted_probs, axis=1) + 1
    true_states = Pred_data['Observed_state'].values

    # Evaluation of accuracy of preditions
    correct = (pred_states == true_states).sum()
    accuracy = correct / len(true_states)

    return -accuracy




# This function corresponds to the D&C method for multitate classification but rescaled with the standard
# deviation of the predicted probabilities (from the training set)
def vectorized_discrepancy_model_std(c_vec, Pred_data_0):
    Pred_data = Pred_data_0.copy().reset_index(drop=True)

    # Extracting probability matrix 
    probs = Pred_data[['Prob_land_state_1', 'Prob_land_state_2', 'Prob_land_state_3']].values
    
    # Computing standard deviations and making sure to avoid division by zero
    stds = np.std(probs, axis=0)
    stds[stds == 0] = 1e-12

    # Standardisation + discrepancy criteria
    adjusted_scores = (probs - c_vec) / stds

    # Prediction and evaluation of accuracy of preditions
    pred_states = np.argmax(adjusted_scores, axis=1) + 1
    true_states = Pred_data['Observed_state'].values

    correct = (pred_states == true_states).sum()
    accuracy = correct / len(true_states)

    return -accuracy






# This function corresponds to the proposed OMCC for multitate classification

def vectorized_discrepancy_model_MCC(c_vec, Pred_data_0):
    Pred_data = Pred_data_0.copy().reset_index(drop=True)
    
    # Applying discrepancy criteria and get predicted class (1-based)
    adjusted_probs = np.stack([
        Pred_data['Prob_land_state_1'].values - c_vec[0],
        Pred_data['Prob_land_state_2'].values - c_vec[1],
        Pred_data['Prob_land_state_3'].values - c_vec[2]
    ], axis=1)
    
    pred_states = np.argmax(adjusted_probs, axis=1) + 1
    true_states = Pred_data['Observed_state'].values
    
    # Building a 3x3 confusion matrix
    confusion_matrix = np.zeros((3, 3), dtype=int)
    for t, p in zip(true_states, pred_states):
        confusion_matrix[t-1, p-1] += 1

    n = confusion_matrix.sum()
    sum_diag = np.trace(confusion_matrix)
    row_sums = confusion_matrix.sum(axis=1)
    col_sums = confusion_matrix.sum(axis=0)

    numerator = n * sum_diag - np.dot(row_sums, col_sums)
    denom_part1 = n**2 - np.sum(row_sums**2)
    denom_part2 = n**2 - np.sum(col_sums**2)
    denominator = np.sqrt(denom_part1 * denom_part2)

    # MCC function 
    MCC = numerator / (denominator + 1e-12)  # MCC function to avoid divide by zero

    return -MCC




# This function corresponds to the proposed OMCC for multitate classification but rescaled with the standard
# deviation of the predicted probabilities (from the training set)

def vectorized_discrepancy_model_MCC_std(c_vec, Pred_data_0):
    Pred_data = Pred_data_0.copy().reset_index(drop=True)
    
    # Computing standard deviations of predicted probabilities (from training set)
    stds = np.array([
        Pred_data['Prob_land_state_1'].std(),
        Pred_data['Prob_land_state_2'].std(),
        Pred_data['Prob_land_state_3'].std()
    ])
    
    # To avoid division by zero in standardisation
    stds[stds == 0] = 1e-12
    
    # Standardising and apply discrepancy criteria
    adjusted_scores = np.stack([
        (Pred_data['Prob_land_state_1'].values - c_vec[0]) / stds[0],
        (Pred_data['Prob_land_state_2'].values - c_vec[1]) / stds[1],
        (Pred_data['Prob_land_state_3'].values - c_vec[2]) / stds[2]
    ], axis=1)
    
    pred_states = np.argmax(adjusted_scores, axis=1) + 1
    true_states = Pred_data['Observed_state'].values

    # Building confusion matrix (3x3)
    confusion_matrix = np.zeros((3, 3), dtype=int)
    for t, p in zip(true_states, pred_states):
        confusion_matrix[t-1, p-1] += 1

    n = confusion_matrix.sum()
    sum_diag = np.trace(confusion_matrix)
    row_sums = confusion_matrix.sum(axis=1)
    col_sums = confusion_matrix.sum(axis=0)

    numerator = n * sum_diag - np.dot(row_sums, col_sums) 
    denom_part1 = n**2 - np.sum(row_sums**2)
    denom_part2 = n**2 - np.sum(col_sums**2)

    # To handle issue in the denominator handling
    if denom_part1 == 0 and denom_part2 == 0:
        denominator = 1
    elif denom_part1 == 0:
        denominator = np.sqrt(1 * denom_part2)
    elif denom_part2 == 0:
        denominator = np.sqrt(denom_part1 * 1)
    else:
        denominator = np.sqrt(denom_part1 * denom_part2)

    # MCC function 
    MCC = numerator / (denominator + 1e-12)
    return -MCC








#### Example use case

- We first simulate a data where the response variable has 3 states (1,2,3) and provide an illustration of the multistate classification algorithm
- The number of states can be made arbitrary (but has to be finite number of states)

In [39]:
np.random.seed(42)

n_individuals_new = 10000
obs_per_individual_new = np.random.randint(1, 6, size=n_individuals_new)
n_obs = obs_per_individual_new.sum()

# covariates (same as before)
X1 = np.random.uniform(-2, 2, size=n_obs)
X2 = np.random.choice([0, 1], size=n_obs)

individual_ids_new = np.repeat(np.arange(n_individuals_new), obs_per_individual_new)

df = pd.DataFrame({"X1": X1, "X2": X2, "sample_id": individual_ids_new})
X_new = sm.add_constant(df[["X1", "X2"]]).to_numpy()  # (n_obs, p)


# Selecting true coefficients
W = np.array([
    [0.5, 0.5, -1.2],   # intercepts effects for classes 1,2,3
    [1.5, -0.8, 1.3],   # covariate X1 effects
    [-1.438,  4.847, -0.444],   # covariates X2 effects
])  

# # ----------------------------------------------
# # if you wanted 4 classes with 2 covariates (for example, one intercept with covariates X1 and X2),
# # you may set:

# W = np.array([
#     [0.5, 0.5, -1.2, 2.5],   # intercepts effects for classes 1,2,3,4
#     [1.5, -0.8, 1.3, -0.9],   # covariate X1 effects
#     [-1.438,  4.847, -0.444, 1.1],   # covariates X2 effects
# ])  


# # ----------------------------------------------


# linear predictors in a data of shape (n_obs, 3)
scores = X_new @ W                    

# On each row, we substrack the maximum value for numeric stability
scores = scores - scores.max(1, keepdims=True)  # we can do this since for softmax function e^{x_i-c}/(sum_j e^{x_j-c}) = e^{x_i}/(sum_j e^{x_j})

# Exponential of scores
P = np.exp(scores)

# Normalising to generate probabilities
P = P / P.sum(1, keepdims=True)   # to make sure each row sums up to 1

# Simulate the states based on P and the number of states (corresponding to len(W))
y = np.array([np.random.choice(len(W), p=p) for p in P])+1  # state \in {1,2,3}

# Setting the response variable column
df["Observed_state"] = y

# Extracting IDs and unique count of IDs
unique_ids_new, groups_new = np.unique(df["sample_id"].values, return_inverse=True) # helps to assign unique
# correct unique to individual with multiple events

n_individuals_new = len(unique_ids_new)


# Adding column of ones
dff = pd.DataFrame(sm.add_constant(df))

dff.columns = ['const' ,'X1', 'X2', 'sample_id', 'Observed_state']

# Reordering columns

dff = dff[['sample_id', 'const', 'X1', 'X2','Observed_state']]

dff

# Setting up indicators (i.e. Binary response variables)
dff['to_state_1'] = (dff['Observed_state']==1).astype('int')
dff['to_state_2'] = (dff['Observed_state']==2).astype('int')
dff['to_state_3'] = (dff['Observed_state']==3).astype('int')


unique_custs = dff['sample_id'].unique()
tp_cust = np.random.choice(unique_custs, int(.75*len(unique_custs))) # selecting 75% of customer as part of
# the training set

# Setting up training and test set (for the puropse of illustration)
tp_train_dat = dff[dff['sample_id'].isin(tp_cust)]
tp_test_dat = dff[~dff['sample_id'].isin(tp_cust)]

# Making temporary copy of y in the training set and tet set
tp_y_observed_train = tp_train_dat['Observed_state'].values
tp_y_observed_test = tp_test_dat['Observed_state'].values

# Only training and test covariates
X_train_dat = tp_train_dat[['const', 'X1','X2']]
X_test_dat = tp_test_dat[['const', 'X1','X2']]

In [40]:
df.columns

Index(['X1', 'X2', 'sample_id', 'Observed_state'], dtype='object')

In [44]:
# Setting instances of the LLink
model_logis_td_state_1 = LogisticRegression(penalty="none")
model_logis_td_state_2 = LogisticRegression(penalty="none")
model_logis_td_state_3 = LogisticRegression(penalty="none")


# Extracting response vars
y_train_to_1 = tp_train_dat['to_state_1'].values
y_train_to_2 = tp_train_dat['to_state_2'].values
y_train_to_3 = tp_train_dat['to_state_3'].values

# Fitting LLink model to each transition-type
model_logis_td_state_1.fit(X_train_dat.values,y_train_to_1)
model_logis_td_state_2.fit(X_train_dat.values,y_train_to_2)
model_logis_td_state_3.fit(X_train_dat.values,y_train_to_3)

# Extracting predicted probabilities on the train set

predicted_probs_to_state_1_train = model_logis_td_state_1.predict_proba(X_train_dat.values)[:,1]
predicted_probs_to_state_2_train = model_logis_td_state_2.predict_proba(X_train_dat.values)[:,1]
predicted_probs_to_state_3_train = model_logis_td_state_3.predict_proba(X_train_dat.values)[:,1]

# Extracting predicted probabilities on the test set
predicted_probs_to_state_1_test = model_logis_td_state_1.predict_proba(X_test_dat.values)[:,1]
predicted_probs_to_state_2_test = model_logis_td_state_2.predict_proba(X_test_dat.values)[:,1]
predicted_probs_to_state_3_test = model_logis_td_state_3.predict_proba(X_test_dat.values)[:,1]

# Concatenating predicted probabilities to relevant data and NAMING APPROPRIATELY SO THAT IT MATCHES THE 
# COLUMN NAMES AND NUMBER OF STATES IN THE MULTISTATE CLASSIFICATION FUNCTIONS

X_train_dat['Prob_land_state_1'] = predicted_probs_to_state_1_train
X_train_dat['Prob_land_state_2'] = predicted_probs_to_state_2_train
X_train_dat['Prob_land_state_3'] = predicted_probs_to_state_3_train


X_test_dat['Prob_land_state_1'] = predicted_probs_to_state_1_test
X_test_dat['Prob_land_state_2'] = predicted_probs_to_state_2_test
X_test_dat['Prob_land_state_3'] = predicted_probs_to_state_3_test


# Concatenating back the response variabls

X_train_dat['Observed_state'] = tp_y_observed_train
X_test_dat['Observed_state'] = tp_y_observed_test

/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/ced001/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_80792/2966332807.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

**Below I print the proportion of each observed state in the training set and test set**

In [128]:
print('Proportion of events in each class observed in training set:\n')

X_train_dat['Observed_state'].value_counts(normalize=True)

Proportion of events in each class observed in training set:



Observed_state
2    0.725997
1    0.233934
3    0.040069
Name: proportion, dtype: float64

In [129]:
print('Proportion of events in each class observed in test set:\n')

X_test_dat['Observed_state'].value_counts(normalize=True)

Proportion of events in each class observed in test set:



Observed_state
2    0.726174
1    0.233981
3    0.039845
Name: proportion, dtype: float64


- We use a **diffential evolution optimisation algorithm** for the estimation of the thresholds vector
- This is an efficient algorithm to find global solutions

In [51]:
from scipy.optimize import differential_evolution

### 4.2. Working example for multistate classification using D&C approach

**The optimisation below is equivalent to the discrepancy decision rule (2.25) for D&C in my thesis**

In [121]:
# Selecting relevant columns
Pred_data0_train = X_train_dat[['Prob_land_state_1','Prob_land_state_2','Prob_land_state_3','Observed_state']]


# Running optimisation
result_train = differential_evolution(
    vectorized_discrepancy_model, # Put the selected multiclass approach (function) here
    bounds=[(0,1)]*3,# to make sure the optimal threshold remain within (0,1)
    args=(Pred_data0_train,),# Input money
    maxiter=500, popsize=15, polish=True, seed=42, disp = True
)

best_DC = -result_train.fun
best_thresholds_DC = result_train.x


differential_evolution step 1: f(x)= -0.8785242570485141
differential_evolution step 2: f(x)= -0.8785877571755144
Polishing solution with 'L-BFGS-B'


In [122]:
print('Accuracy of the training set is:', np.round(100*best_DC,3),'%.')


Accuracy of the training set is: 87.859 %.


In [123]:
print('The optimal vector of thresholds is:',best_thresholds_DC)

The optimal vector of thresholds is: [0.51947368 0.73030717 0.63974526]


**Using the above optimal thresholds, we can evaluate the accuracy on the test set**
- N.B. Since we used the D&C (without rescaling) multistate approach, we apply the same the corresponding discrepancy measure (this is equivalent to equation (2.23) in the thesis)

In [124]:
# Setting up data
Pred_data0_test = X_test_dat[['Prob_land_state_1','Prob_land_state_2','Prob_land_state_3','Observed_state']]

# Comupting the discrepancy scores for D&C 
adjusted = Pred_data0_test.iloc[:, :-1].to_numpy() - best_thresholds_DC

# Predicting next landing state
pred_states_test_set = np.argmax(adjusted, axis=1) + 1


#-------------- Measuring overall accuracy of predictions (i.e. to all states)

# Adding the predicted state as column to the test set
Pred_data0_test['Predicted_state'] = pred_states_test_set

# Measuring the accuracy of predictions
print('Overall accuracy of prediction to next state on test set using D&C is:',
      np.round(100*(Pred_data0_test['Observed_state']==Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 1

# Selecting rows event where the observed state is 1
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==1] 

print('Accuracy of prediction to state 1 on test set using  D&C is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 2

# Selecting rows event where the observed state is 2
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==2] 

print('Accuracy of prediction to state 2 on test set using D&C is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 3

# Selecting rows event where the observed state is 3
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==3] 

print('Accuracy of prediction to state 3 on test set is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')

Overall accuracy of prediction to next state on test set using D&C is: 87.828 %


Accuracy of prediction to state 1 on test set using  D&C is: 84.179 %


Accuracy of prediction to state 2 on test set using D&C is: 93.822 %


Accuracy of prediction to state 3 on test set is: 0.0 %


/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_80792/1235245185.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Pred_data0_test['Predicted_state'] = pred_states_test_set


### 4.3. Working example for multistate classification using our approach (OMCC)


**We now estimate the optimal thresholds using the multistate classification approach we proposed in the thesis (i.e. the OMCC)**

In [116]:
# Selecting relevant columns
Pred_data0_train = X_train_dat[['Prob_land_state_1','Prob_land_state_2','Prob_land_state_3','Observed_state']]


# Running optimisation
result_train = differential_evolution(
    vectorized_discrepancy_model_MCC, # Put the selected multiclass approach (function) here
    bounds=[(0,1)]*3,# to make sure the optimal threshold remain within (0,1)
    args=(Pred_data0_train,),# Input money
    maxiter=500, popsize=15, polish=True, seed=42, disp = True
)

best_OMCC = -result_train.fun
best_thresholds_OMCC = result_train.x


differential_evolution step 1: f(x)= -0.7061168270542098
differential_evolution step 2: f(x)= -0.7061168270542098
differential_evolution step 3: f(x)= -0.7061168270542098
differential_evolution step 4: f(x)= -0.7062384440472144
Polishing solution with 'L-BFGS-B'


In [117]:
print('MCC **SCORE** on the training set is:', np.round(100*best_OMCC,3),'%.')

MCC **SCORE** on the training set is: 70.624 %.


In [118]:
print('The optimal vector of thresholds is: on the training set is:', best_thresholds_OMCC)

The optimal vector of thresholds is: on the training set is: [0.18376418 0.56258003 0.32031983]


In [120]:
# Setting up data
Pred_data0_test = X_test_dat[['Prob_land_state_1','Prob_land_state_2','Prob_land_state_3','Observed_state']]

# Comupting the discrepancy scores for D&C 
adjusted = Pred_data0_test.iloc[:, :-1].to_numpy() - best_thresholds_OMCC

# Predicting next landing state
pred_states_test_set = np.argmax(adjusted, axis=1) + 1


#-------------- Measuring overall accuracy of predictions (i.e. to all states)

# Adding the predicted state as column to the test set
Pred_data0_test['Predicted_state'] = pred_states_test_set

# Measuring the accuracy of predictions
print('Overall accuracy of prediction to next state on test set using the OMCC is:',
      np.round(100*(Pred_data0_test['Observed_state']==Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 1

# Selecting rows event where the observed state is 1
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==1] 

print('Accuracy of prediction to state 1 on test set using the OMCC  is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 2

# Selecting rows event where the observed state is 2
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==2] 

print('Accuracy of prediction to state 2 on test set using the OMCC  is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')
print('\n')

#-------------- Measuring accuracyof predictions to state 3

# Selecting rows event where the observed state is 3
tp_Pred_data0_test = Pred_data0_test[Pred_data0_test['Observed_state']==3] 

print('Accuracy of prediction to state 3 on test set using the OMCC is:',
      np.round(100*(tp_Pred_data0_test['Observed_state']==tp_Pred_data0_test['Predicted_state']).mean(),3),'%')

Overall accuracy of prediction to next state on test set using the OMCC is: 87.171 %


Accuracy of prediction to state 1 on test set using the OMCC  is: 87.017 %


Accuracy of prediction to state 2 on test set using the OMCC  is: 92.003 %


Accuracy of prediction to state 3 on test set using the OMCC is: 0.0 %


/var/folders/2w/qgtm8nnx5k992f8ws4bm58940000gn/T/ipykernel_80792/1813275817.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Pred_data0_test['Predicted_state'] = pred_states_test_set
